In [ ]:
#! pip install transformers datasets evaluate accelerate

# Text classification

In [ ]:
import os
from dotenv import load_dotenv
from huggingface_hub import login
#from huggingface_hub import notebook_login

load_dotenv()
login(token=os.getenv("HF_TOKEN"))
#notebook_login()

## Load ccdv/arxiv-summarization dataset

https://huggingface.co/datasets/ccdv/arxiv-summarization


In [ ]:
from datasets import load_dataset

dataset = load_dataset("ccdv/arxiv-summarization")

In [ ]:
print("Columns:", dataset['train'].column_names)

In [ ]:
sample = dataset['train'][0]
print("\n--- (Input) ---")
print(sample['article'][:5000], "...\n")
print("--- (Output) ---")
print(sample['abstract'])

## Preprocess

The next step is to load a DistilBERT tokenizer to preprocess the `text` field:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("google-t5/t5-small")
task = "summarize: "

Create a preprocessing function to tokenize `text` and truncate sequences to be no longer than DistilBERT's maximum input length:

In [ ]:
def preprocesar_seq2seq(samples):
    inputs = [task + doc for doc in samples["article"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)
    labels = tokenizer(text_target=samples["abstract"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [ ]:
tokenized_dataset = dataset.map(preprocesar_seq2seq, batched=True)
print("Columns ready for training:", tokenized_dataset['train'].column_names)

In [ ]:
#! pip install rouge_score

In [ ]:
import numpy as np
import evaluate
from transformers import (
    DataCollatorForSeq2Seq, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer
)
model = AutoModelForSeq2SeqLM.from_pretrained("google-t5/t5-small")
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
rouge = evaluate.load("rouge")

## Evaluate

In [ ]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v * 100, 4) for k, v in result.items()}

## Train

In [ ]:
import itertools
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

train_subset = tokenized_dataset["train"].shuffle(seed=42).select(range(20000))
val_subset = tokenized_dataset["validation"].shuffle(seed=42).select(range(2000))

training_args = Seq2SeqTrainingArguments(
    output_dir="t5-arxiv-summarization-final",
    per_device_train_batch_size=4,       
    per_device_eval_batch_size=4,
    learning_rate=5e-5,                  
    weight_decay=0.01,
    num_train_epochs=5,                  
    eval_strategy="epoch",
    save_strategy="epoch",
    predict_with_generate=True,          
    fp16=True,                           
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",      
    disable_tqdm=False
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_subset,
    eval_dataset=val_subset,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

# 4. Lanzar el entrenamiento
print("Launching abstractive summarization training pipeline...")
trainer.train()

In [ ]:
trainer.push_to_hub()

## Inference

In [ ]:
import torch
from transformers import AutoTokenizer, pipeline

paper_sample = dataset['test'][0]['article']
print("--- ORIGINAL ARTICLE ---")
print(paper_sample[:500], "...\n")

In [ ]:
model_name = "jclondonol/t5-arxiv-summarization-final"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
input_text = "summarize: " + paper_sample
inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt")
truncated_text = tokenizer.decode(inputs["input_ids"][0], skip_special_tokens=True)

In [ ]:
summarizer = pipeline(
    "text-generation", 
    model=model_name,
    device=0 if torch.cuda.is_available() else -1
)
print("[INFO] Running inference on bounded token space...")
print("-" * 65)

In [ ]:
prediction = summarizer(
    truncated_text,
    max_new_tokens=150,  
    clean_up_tokenization_spaces=True
)

print("--- SUMMARY GENERATED BY THE MODEL ---")
raw_output = prediction[0]['generated_text']

clean_summary = raw_output.replace(truncated_text, "").strip()
print(clean_summary if clean_summary else raw_output)

In [ ]:

raw_output = prediction[0]['generated_text']

print("=" * 65)
print("CLEAN ABSTRACTIVE SUMMARY GENERATED BY T5")
print("=" * 65)

if raw_output.startswith(input_text):
    clean_summary = raw_output[len(input_text):].strip()
elif raw_output.startswith("summarize:"):
    clean_summary = raw_output[len("summarize:"):].strip()
else:
    clean_summary = raw_output.strip()

print(clean_summary)
print("=" * 65)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "jclondonol/t5-arxiv-summarization-final"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

paper_sample = dataset['test'][0]['article']
input_text = "summarize: " + paper_sample

inputs = tokenizer(input_text, max_length=1024, truncation=True, return_tensors="pt").to(device)

print("[INFO] Processing text through strict Encoder-Decoder generation...")
print("-" * 65)

with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=100,       
        min_length=30,        
        num_beams=4,          
        no_repeat_ngram_size=3, 
        early_stopping=True
    )


clean_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("--- GENUINE ABSTRACTIVE SUMMARY ---")
print(clean_summary)
print("=" * 65)